# 9. Is the higher flagging of a group explained by the prevalence of disease in the fitting data?

A group may be flagged more often because the disease was more common in it in the fitting data, or because its ECGs look more abnormal for reasons that the labels do not capture. To test the first explanation, the composite SHD detectors are **refitted on a subsample of the fitting ECGs** of every round:

* within every stratum of age band × sex, each race and ethnicity group (the four groups, and patients recorded as other or unknown) keeps all ECGs of one class and a random subset of the other, so that its prevalence of composite SHD equals that of the stratum ("equalised");
* a detector fitted on a **random subsample of the same size** that keeps the prevalences of the fitting data controls for the loss of fitting data ("random subsample");
* a detector fitted on all fitting ECGs ("all rows"), so that the three fits are compared with the same solver.

Validation (C and the 80%-sensitivity threshold) and evaluation are unchanged. Differences from White patients are computed as in notebook 07, with 1000 patient-bootstrap resamples (paired across the three fits).

**Paper items:** Supplementary Fig. 3, Table 2 (question 5), and the share of fitting ECGs kept (89% in EchoNext, 92% in MIMIC-IV).

Run time: a few minutes on one RTX 5090.

In [ ]:
%run ./00_settings.ipynb
OUT = results("09_prevalence_equalised_refit")
VARIANTS = ["all rows", "equalised", "random subsample"]


def equalise(y, g, s, rng):
    """Boolean mask over fitting rows: within every stratum s, each group g keeps all rows of one class and a random part of the other, so that
    its prevalence equals that of the stratum."""
    keep = np.zeros(len(y), bool)
    for st in np.unique(s):
        m = s == st
        p = y[m].mean()
        for grp in np.unique(g[m]):
            pos, neg = np.flatnonzero(m & (g == grp) & (y == 1)), np.flatnonzero(m & (g == grp) & (y == 0))
            if len(pos) == 0 or len(neg) == 0 or p in (0, 1):
                keep[pos] = keep[neg] = True                       # nothing to balance in this cell
                continue
            if len(pos) / (len(pos) + len(neg)) > p:               # too many positives: thin them out
                pos = rng.choice(pos, int(round(p / (1 - p) * len(neg))), replace=False)
            else:
                neg = rng.choice(neg, int(round((1 - p) / p * len(pos))), replace=False)
            keep[pos] = keep[neg] = True
    return keep


def run(cohort, X_of, rows, pi_of_row, g_row, y_row, g_lat, y_lat, s_lat):
    out, sizes = [], []
    sel = g_lat >= 0
    g, y, s = g_lat[sel], y_lat[sel], s_lat[sel]
    n_pat = len(g_lat)
    for fm in [k for k in PAPER_ORDER]:
        t0 = time.time()
        X = X_of(fm)
        PR = {v: np.zeros(n_pat, np.int8) for v in VARIANTS}
        P = {v: np.full(n_pat, np.nan) for v in VARIANTS}
        rng = np.random.default_rng(SEED_REFIT)
        for k, (tr, va, ev) in enumerate(rows):
            Xs = {n: to_dev(X[i]) for n, i in (("tr", tr), ("va", va), ("ev", ev))}
            standardize_(Xs["tr"], Xs["va"], Xs["ev"])                # the same standardisation for the three fits
            ytr, yva = y_row[tr].astype(int), y_row[va].astype(int)
            gtr, str_ = g_row[tr], s_lat[pi_of_row[tr]]              # stratum of the patient (age and sex at the most recent ECG)
            eq = equalise(ytr, gtr, str_, rng)
            rs = np.zeros(len(tr), bool); rs[rng.choice(len(tr), int(eq.sum()), replace=False)] = True
            pe = pi_of_row[ev]
            for v, mask in (("all rows", np.ones(len(tr), bool)), ("equalised", eq), ("random subsample", rs)):
                b = lr_select(Xs["tr"][torch.as_tensor(mask, device=device())], ytr[mask], Xs["va"], yva, C_DETECTOR)
                thr = float(np.quantile(sigmoid(lr_scores(Xs["va"], b["W"], b["b"]))[yva == 1], 1 - SENS))
                p = sigmoid(lr_scores(Xs["ev"], b["W"], b["b"]))
                P[v][pe], PR[v][pe] = p, (p >= thr).astype(np.int8)
                if fm == PAPER_ORDER[0]:
                    sizes.append({"cohort": cohort, "fold": k, "variant": v, "rows": int(mask.sum()), "rows_all": int(len(tr)),
                                  **{f"prev_{RACE4[gi]}": float(ytr[mask & (gtr == gi)].mean()) for gi in range(4)}})
            del Xs
            free_gpu()
        stats = {v: error_differences(y, PR[v][sel], g, s)[0] for v in VARIANTS}
        rng = np.random.default_rng(SEED_REFIT)
        boot = {v: np.empty((N_BOOT,) + stats[v].shape) for v in VARIANTS}
        for b_ in range(N_BOOT):
            ix = rng.integers(0, len(y), len(y))
            for v in VARIANTS:
                boot[v][b_] = error_differences(y[ix], PR[v][sel][ix], g[ix], s[ix])[0]
        for v in VARIANTS:
            for pi, (a_, b2) in enumerate(PAIRS):
                for mi, meas in enumerate(MEASURES):
                    d, bd = stats[v][pi, mi], boot[v][:, pi, mi]
                    dd = bd - boot["all rows"][:, pi, mi]
                    out.append({"cohort": cohort, "fm": FMS[fm], "fm_key": fm, "variant": v, "pair": f"{a_}-{b2}", "measure": meas,
                                "auroc": fast_auc(y_lat, P[v]), "diff": d, "lo": ci95(bd)[0], "hi": ci95(bd)[1],
                                "change_vs_all_rows": d - stats["all rows"][pi, mi], "change_lo": ci95(dd)[0], "change_hi": ci95(dd)[1]})
        print(f"[{cohort}] {fm}: {(time.time() - t0) / 60:.1f} min")
    return out, sizes

## 9.1 EchoNext and MIMIC-IV

In [ ]:
D = load_design("echonext")
j = E_LABELS.index(E_COMP)
s_e = strata6(D["age"], D["male"])
res_e, size_e = run("EchoNext", emb_echonext, [fold_rows(D, k) for k in range(N_FOLDS)], D["pi_of_row"], D["g"][D["pi_of_row"]], D["Y_row"][:, j],
                    D["g"], D["Y"][:, j], s_e)

C = load_mimic_cohort()
Dm = load_design("mimic")
lat_m = C.loc[Dm["lat_idx"]].reset_index(drop=True)
rows_m = C.row.to_numpy()
res_m, size_m = run("MIMIC-IV", lambda fm: emb_mimic(fm)[rows_m], [fold_rows(Dm, k) for k in range(N_FOLDS)], Dm["pi_of_row"], race_index(C.race6),
                    C[M_COMP].to_numpy(float), race_index(lat_m.race6), lat_m[M_COMP].to_numpy(float),
                    strata6(lat_m.age.to_numpy(float), lat_m.male.to_numpy(int)))
summary = pd.DataFrame(res_e + res_m)
sizes = pd.DataFrame(size_e + size_m)
summary.to_csv(OUT / "summary.csv", index=False, encoding="utf-8-sig")
sizes.to_csv(OUT / "fitting_rows.csv", index=False, encoding="utf-8-sig")
kept = sizes[sizes.variant == "equalised"].groupby("cohort")[["rows", "rows_all"]].sum()
print("share of fitting ECGs kept by the equalised subsample:", (kept.rows / kept.rows_all).round(3).to_dict())
q = summary[(summary.measure == "FLAG") & summary.pair.isin(["black-white", "asian-white"])]
q.assign(diff=100 * q["diff"], lo=100 * q.lo, hi=100 * q.hi)[["cohort", "pair", "fm", "variant", "diff", "lo", "hi"]].round(1)